# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 3: *Feature Engineering*
##### Version Number: 2.0
---
### Contents  
> 1. *Load Data*
> 2. *Days Without Rain*
> 3. *Fire History*
> 4. *Lagged Variables*
> 5. *Export Data*
---
### Notes
> Features Added:
> - `Days_without_Rain` A rolling count that serves as a simple drought indicator
> - `Fire_History` An average count of fires per month in a region spanning the alst two years (needs refinement)
> - `Lagged_Variables` 7 day rolling averages for 'Avg Air Temp (F)', 'Precip (in)', 'Avg Rel Hum (%)', 'Avg Wind Speed (mph)'
---
### Inputs
- `trimmed.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `final.csv` - final clean dataset with features added
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

# Function to print a custom format correlation heatmap
from src.plot_utils import correlation_map

# Function to calculate dryness index and return a dataframe
from src.data_utils import calculate_dryness_index

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize


# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

## 1. Load Datasets

In [3]:
# Load cleaned main dataset
final = pd.read_csv("../data/processed/fire_pop_income.csv")

In [4]:
final.columns

Index(['Sample_ID', 'Date', 'Sample_Longitude', 'Sample_Latitude',
       'Sample_Elevation', 'Region_ID', 'Region_Name', 'Stn_Id', 'Stn_Name',
       'County', 'geometry_x', 'Stn Name', 'CIMIS Region', 'ETo (in)',
       'Precip (in)', 'Sol Rad (Ly/day)', 'Avg Vap Pres (mBars)',
       'Max Air Temp (F)', 'Min Air Temp (F)', 'Avg Air Temp (F)',
       'Max Rel Hum (%)', 'Min Rel Hum (%)', 'Avg Rel Hum (%)',
       'Dew Point (F)', 'Avg Wind Speed (mph)', 'Wind Run (miles)',
       'Avg Soil Temp (F)', 'Season', 'fire_count', 'total_fire_damage',
       'Total Population', 'density', 'Mean Income', '_merge', 'Target'],
      dtype='object')

In [5]:
final

,Sample_ID,Date,Sample_Longitude,Sample_Latitude,Sample_Elevation,Region_ID,Region_Name,Stn_Id,Stn_Name,County,...,Wind Run (miles),Avg Soil Temp (F),Season,fire_count,total_fire_damage,Total Population,density,Mean Income,_merge,Target
0,1,2018-01-01,-117.232017,32.778837,36.609302,7,Marine Region,173,Torrey Pines,San Diego,...,74.6,56.8,0,0,0.0,3269973,777.337917,111241,both,0
1,33,2018-01-01,-119.732017,34.778837,1096.443481,5,South Coast Region,88,Cuyama,Santa Barbara,...,98.3,53.8,0,0,0.0,441257,161.331803,111731,both,0
2,144,2018-01-01,-121.732017,40.278837,1039.047485,1,Northern Region,222,Gerber South,Tehama,...,81.2,47.0,0,0,0.0,64896,22.000807,74237,both,0
3,32,2018-01-01,-120.232017,34.778837,288.821625,5,South Coast Region,165,Sisquoc,Santa Barbara,...,94.7,61.8,0,0,0.0,441257,161.331803,111731,both,0
4,145,2018-01-01,-121.232017,40.278837,1615.276733,2,North Central Region,12,Durham,Butte,...,43.3,45.9,0,0,0.0,207172,126.597656,79233,both,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447775,52,2025-01-23,-116.732017,35.278837,925.951294,6,Inland Deserts Region,117,Victorville,San Bernardino,...,118.5,40.6,0,0,0.0,2195611,109.468892,85327,both,0
447776,124,2025-01-23,-123.232017,39.278837,473.016693,1,Northern Region,106,Sanel Valley,Mendocino,...,60.8,46.0,0,0,0.0,89108,25.413394,74102,both,0
447777,51,2025-01-23,-117.232017,35.278837,1046.279663,6,Inland Deserts Region,117,Victorville,San Bernardino,...,118.5,40.6,0,0,0.0,2195611,109.468892,85327,both,0
447778,50,2025-01-23,-117.732017,35.278837,956.628479,4,Central Region,257,Ridgecrest,Kern,...,42.3,49.7,0,0,0.0,913820,112.374445,75161,both,0


---

## 2. Days Without Rain

flag to record whether **signifigant ( >1 inch)** precipitation has occured on a day

In [6]:
stations = final['Sample_ID'].unique()
final['Days Without Rain'] = 0
day_count = 0;

In [7]:
stations

array([  1,  33, 144,  32, 145,  31, 146,  30, 147,  29, 148,  28, 149,
        27, 150,  26, 151,  25, 143,  34, 142,  35, 133,  44, 134,  43,
       135,  42, 136,  41, 152, 137, 138,  39,  38, 139,  37, 140,  36,
       141,  40,  45,  24,  23,  11, 165,  10, 166,   9, 167,   8, 168,
         7, 169,   6, 170,   5, 171,   4, 172,   3, 164,  12, 163,  13,
        22, 154,  21, 155,  20, 156,  19, 157, 153,  18,  17, 159,  16,
       160,  15, 161,  14, 162, 158, 132,  46, 131,  78,  77, 102,  76,
       103,  75, 104,  74, 105,  73, 106,  72, 107,  71, 108,  70, 109,
       101,  79, 100,  80,  89,  91,  88,  92,  87,  93,  86,  94,  69,
        85,  84,  96,  83,  97,  82,  98,  81,  99,  95, 110,  68, 111,
       122,  55, 123,  54, 124,  53, 125,  52,  56, 126, 127,  50, 128,
        49, 129,  48, 130,  47,  51, 173, 121,  58,  67, 112,  66, 113,
        65, 114,  64, 115,  57,  63,  62, 117,  61, 118,  60, 119,  59,
       120, 116,   2,  90], dtype=int64)

In [8]:
for station in stations:
    station_indexes = final[final['Sample_ID'] == station].index
    day_count=0
    for index, row in final.loc[station_indexes].iterrows():
        if row['Precip (in)'] > 0:
            day_count = 0
        else:
            day_count = day_count + 1
    
        final.at[index, 'Days Without Rain'] = day_count


In [9]:
final[final['Sample_ID'] == 2]['Days Without Rain']

171        1
198        0
381        1
601        0
804        0
          ..
446990    37
447144    38
447431    39
447552    40
447691    41
Name: Days Without Rain, Length: 2585, dtype: int64

---

## 3.  Historical fires per month

Create a new column that records the historical average number of fires for each month, based on all previous years up to that point. Each row should reflect the average number of fires that occurred in the same calendar month across prior years, providing a historical baseline for comparison.

**ASSUMPTION** -Since the weather coverage in this dataset is limited, no previous year readings exists.
For now, I will just use the current count as a rough estimation for the average. 

In [10]:
final['Date'] = pd.to_datetime(final['Date'])

# Convert Datetime column to string
final['Month_Year'] = final['Date'].dt.strftime('%Y-%m')
month_year = final['Month_Year'].unique()

In [11]:
# Dictionary to store month-to-count mapping
monthly_fire_counts = {}

# Step 1: Calculate fire counts for each month
for month in month_year:
    # Get index and subset
    month_index = final[final['Month_Year'] == month].index
    subset = final.loc[month_index]
    
    # Eliminate 'Low' days
    total = subset[subset['Target'] != 0]
    
    # Store count
    monthly_fire_counts[month] = len(total)

# Step 2: Calculate 2-year rolling average
# Convert month_year to sorted list to ensure order
month_year_sorted = sorted(month_year)

# Dictionary for rolling averages
rolling_avg = {}

for i, month in enumerate(month_year_sorted):
    if i == 0:
        # First year: average of all *future* years (exclude current)
        future_values = list(monthly_fire_counts.values())[1:]
        rolling_avg[month] = np.mean(future_values) if future_values else 0

    elif i == 1:
        # Second year: average of first year and current
        prev = month_year_sorted[0]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev], monthly_fire_counts[month]])
    
    else:
        # Rolling average of previous 2 years
        prev1 = month_year_sorted[i - 1]
        prev2 = month_year_sorted[i - 2]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev1], monthly_fire_counts[prev2]])

# Step 3: Assign back to each row in the dataframe
for month in month_year_sorted:
    month_index = final[final['Month_Year'] == month].index
    final.loc[month_index, '2-Year Avg Fires'] = rolling_avg[month]

---

## 4. Add Lagged Weather Variables

We calculate 7-day rolling averages for select weather variables to capture recent trends that may influence fire severity.

In [12]:
# Sort data by County and Date to prepare for rolling average
final = final.sort_values(by=['Sample_ID', 'Date'])

# Define columns for 7-day rolling average
avg_columns = [
    'Avg Air Temp (F)', 
    'Precip (in)',
    'Avg Rel Hum (%)', 
    'Avg Wind Speed (mph)'
]

# Apply rolling mean by County
for col in avg_columns:
    final[f'{col} 7 Day Avg'] = final.groupby('Sample_ID')[col].rolling(window=7, min_periods=1).mean().reset_index(level=0, drop=True)

In [13]:
final['Month_Year'] = pd.to_datetime(final['Month_Year'])
final['Month'] = final['Month_Year'].dt.month
final['Year'] = final['Month_Year'].dt.year

In [14]:
drop = ['Month_Year']
final = final.drop(columns=drop)

In [15]:
drop_cols =['Sample_Longitude', 'Sample_Latitude', 'Region_Name', 'Stn_Id', 'Stn_Name',
       'County', 'geometry_x', 'Stn Name', 'CIMIS Region', 'fire_count', 'total_fire_damage',
        '_merge',]

## drop unneeded columns
model_fire_pop_income = final.drop(columns=drop_cols)

## Save some additional useful columns for further analysis
keep_columns = ['Sample_ID','Date','Region_ID'] 

#  build list of columns to save in details
drop_cols_with_id =  keep_columns + drop_cols

details = final[drop_cols_with_id].copy()

---

## 5. Export Data

In [16]:
model_fire_pop_income.to_csv("../data/processed/model_fire_pop_income.csv", index=False)
details.to_csv('../data/processed/details.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
